In [1]:
import re
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dlcourseproject/nlp-getting-started/train.csv")

In [ ]:
# 1. Load dataset
df = df[['text', 'target']].dropna()

In [ ]:
df

In [ ]:
# 2. Light cleaning for BERT
def clean_for_bert(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['text'] = df['text'].apply(clean_for_bert)

In [ ]:
# 3. Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

In [ ]:
# 4. Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=96
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=96
)

In [ ]:
# 5. Dataset class
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TweetDataset(train_encodings, list(y_train))
test_dataset = TweetDataset(test_encodings, list(y_test))

In [ ]:
# 6. Model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
# 7. Training args
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    report_to="none"
)

In [ ]:
# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [ ]:
# 9. Train
trainer.train()

In [ ]:
# 10. Evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
true = []

for item in test_dataset:
    inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "labels"}
    label = item['labels'].item()

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    preds.append(pred)
    true.append(label)

bert_acc = accuracy_score(true, preds)
print("BERT Accuracy:", bert_acc)
print(classification_report(true, preds))
print(predict_bert("earthquake destroyed buildings"))
print(predict_bert("flood of emotions after breakup"))
print(predict_bert("this match is fire 🔥"))

In [ ]:
# 11. Save model and tokenizer
model.save_pretrained("/content/drive/MyDrive/dlcourseproject/bert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dlcourseproject/bert_model")

print("Model and tokenizer saved successfully!")

In [ ]:
def predict_bert(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    pred = probs[0][1].item()

    if pred > 0.7:
        return "Disaster", pred
    elif pred < 0.3:
        return "Not Disaster", pred
    else:
        return "Uncertain", pred

In [ ]:
def predict_multiple(tweets):
    results = []

    for tweet in tweets:
        label, score = predict_bert(tweet)
        results.append({
            "tweet": tweet,
            "prediction": label,
            "confidence": round(score, 4)
        })

    return results

In [ ]:
tweets = [
    "earthquake destroyed buildings",
    "just chilling with friends",
    "flood in city area",
    "this match is fire 🔥"
]

results = predict_multiple(tweets)

for r in results:
    print(r)